# QCNN-Style QNN for Industrial Micro-Defect Inspection

Your team has already gone deep on QNNs, so this notebook keeps the QNN direction alive but makes it more defensible.

The real-world use case is **industrial micro-defect inspection**: semiconductor wafers, battery materials, aerospace composites, weld scans, or microscopic surface inspection. In these settings, labeled defects are rare, the signal can be a subtle texture-correlation shift, and a generic deep classical model may be too data-hungry.

This notebook implements a compact **QCNN-style** classifier. It is inspired by quantum convolutional neural networks, but scaled down for a hackathon laptop: four compact texture features are encoded onto four qubits, local pair interactions act like convolution, pooling-like gates concentrate pair information, and a final readout qubit predicts clean vs defective.

## 1. Imports

The defaults are intentionally small. If this works, increase the training subset or optimizer iterations later.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers and Synthetic Industrial Proxy

This is not a clinical or manufacturing dataset. It is a controlled proxy for a realistic pattern: compact inspection features where the defect label depends on coupled texture factors rather than one obvious pixel.

In [ ]:
def generate_microdefect_latents(n_samples=80, seed=SEED + 100):
    """Synthetic proxy for calibrated industrial texture features.

    Think of each row as four compact features produced by an inspection front-end:
    two local texture phase measurements and two orientation/defect-coupling scores.
    The label depends on coupled factors, not a single obvious threshold.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 2 * np.pi, size=(n_samples, 4))
    score = np.sin(X[:, 0]) * np.cos(X[:, 1]) + np.sin(X[:, 2] + X[:, 3])
    y = (score > np.median(score)).astype(int)
    return X, y, score


def latents_to_microtexture_images(X, size=8, seed=SEED):
    """Render latent texture factors as small grayscale inspection patches."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)


def show_microtextures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.2f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=18, seed=SEED):
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def readout_q0(measured_integer):
    """Use qubit 0 as the binary classifier readout."""
    return measured_integer & 1


def parity_readout(measured_integer):
    """Map a measured computational-basis state to class 0 or 1 by parity."""
    return bin(measured_integer).count("1") % 2



def build_qcnn_style_circuit(n_qubits=4):
    """A compact QCNN-style circuit for four texture features.

    This is not a full image CNN. It is a small structured QNN inspired by QCNN
    motifs: local convolutional pair interactions, pooling-like information
    concentration, then a final interaction between pooled channels.
    """
    if n_qubits != 4:
        raise ValueError("This compact demo is written for exactly four qubits.")

    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", 10)
    qc = QuantumCircuit(n_qubits)

    for q in range(n_qubits):
        qc.ry(x[q], q)

    # Shared local convolution on pairs (0, 1) and (2, 3).
    qc.cx(0, 1)
    qc.ry(theta[0], 1)
    qc.rz(theta[1], 0)
    qc.cx(1, 0)
    qc.ry(theta[2], 0)

    qc.cx(2, 3)
    qc.ry(theta[0], 3)
    qc.rz(theta[1], 2)
    qc.cx(3, 2)
    qc.ry(theta[2], 2)

    # Pooling-like interactions: concentrate each pair into qubits 0 and 2.
    qc.cx(1, 0)
    qc.ry(theta[3], 0)
    qc.cz(1, 0)

    qc.cx(3, 2)
    qc.ry(theta[4], 2)
    qc.cz(3, 2)

    # Final convolution/readout layer between pooled channels.
    qc.cx(0, 2)
    qc.ry(theta[5], 2)
    qc.rz(theta[6], 0)
    qc.cx(2, 0)
    qc.ry(theta[7], 0)
    qc.ry(theta[8], 2)
    qc.cz(0, 2)
    qc.ry(theta[9], 0)

    return qc, list(x), list(theta)

## 3. Generate Texture Features and Inspection Patches

The four features are interpreted as compact texture/phase measurements from a classical inspection front-end. The images are shown only to keep the classification problem visually grounded.

In [ ]:
N_SAMPLES = 80
X, y, scores = generate_microdefect_latents(n_samples=N_SAMPLES, seed=SEED + 100)
images = latents_to_microtexture_images(X, size=8, seed=SEED)

print("feature shape:", X.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})
show_microtextures(images, y, scores=scores, n_show=12, cols=6)

## 4. Train/Test Split

We stratify the split because rare-defect work should never rely on ordinary accuracy alone.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print("train:", X_train.shape, "clean:", int((y_train == 0).sum()), "defect:", int((y_train == 1).sum()))
print("test :", X_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 5. Classical Baselines

These baselines are important. A QCNN-style model is only interesting if it does something nontrivial relative to simple classical models.

In [ ]:
classical_models = {
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ]),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in classical_models.items():
    model.fit(X_train, y_train)
    evaluate_classifier(name, model, X_test, y_test, show_confusion=False)

## 6. Build the QCNN-Style Circuit

This circuit uses only 10 trainable parameters. That is the point: a structured QNN should avoid the worst behavior of a large random ansatz.

In [ ]:
qcnn_circuit, input_params, weight_params = build_qcnn_style_circuit(n_qubits=4)

print("qubits:", qcnn_circuit.num_qubits)
print("trainable weights:", len(weight_params))
print("circuit depth:", qcnn_circuit.depth())
qcnn_circuit.draw(output="mpl")

## 7. Train the QCNN-Style Classifier

The model uses qubit 0 as the readout. Start with a small balanced subset; then increase `QCNN_TRAIN_PER_CLASS` or `QCNN_MAXITER` if the result is promising.

In [ ]:
QCNN_TRAIN_PER_CLASS = 18
QCNN_MAXITER = 40

X_qcnn_train, y_qcnn_train = balanced_subset(
    X_train,
    y_train,
    n_per_class=QCNN_TRAIN_PER_CLASS,
    seed=SEED,
)

sampler_qnn = SamplerQNN(
    circuit=qcnn_circuit,
    sampler=StatevectorSampler(seed=SEED),
    input_params=input_params,
    weight_params=weight_params,
    interpret=readout_q0,
    output_shape=2,
)

rng = np.random.default_rng(SEED)
initial_point = rng.uniform(-0.2, 0.2, size=len(weight_params))

qcnn_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn,
    optimizer=COBYLA(maxiter=QCNN_MAXITER),
    initial_point=initial_point,
)

qcnn_classifier.fit(X_qcnn_train, y_qcnn_train)
evaluate_classifier("QCNN-style SamplerQNN", qcnn_classifier, X_test, y_test)

## 8. How to Interpret This Direction

This is the best QNN-centered pitch if your team wants to stay with QNNs:

- It is structured, not a random hardware-efficient ansatz.
- It matches local-to-global inspection logic.
- It uses few parameters, which helps with small data.
- It is still a prototype: test several seeds before making a performance claim.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Furman and Sahinidis, "Computational complexity of heat exchanger network synthesis," Computers & Chemical Engineering 25(9-10), 1371-1390 (2001): https://doi.org/10.1016/S0098-1354(01)00681-0
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html